# Notebook 02 - Synthetic Signal Analysis

This notebook analyzes invented values from anonymous signal channels. Results are illustrative and are not sourced from an organization or operating asset.

## Analysis step 2

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── project root on the path so src/ imports work ──────────────────────────
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── plotting style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.constrained_layout.use": True,
})
ACCENT   = "#E84B4B"   # red  – anomaly / shutdown highlight
NEUTRAL  = "#4878D0"   # blue – normal sensor trace
GREY     = "#AAAAAA"   # grey – reference / background

print("Setup complete.")

## Analysis step 3

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
df = pd.read_csv(
    ROOT / "data" / "interim" / "channel_values.csv",
    index_col=0, parse_dates=True
)
channel = pd.read_csv(ROOT / "data" / "interim" / "channel_dictionary.csv")

print(f"Shape  : {df.shape}  →  {df.shape[0]} days × {df.shape[1]} sensors")
print(f"From   : {df.index.min().date()}")
print(f"To     : {df.index.max().date()}")
print(f"\nTag dictionary : {len(channel)} rows")
df.head(3)

## Analysis step 4

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# ── per-column missing rate ────────────────────────────────────────────────
miss = df.isna().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(14, 7))
colors = [ACCENT if v > 50 else ("#FFA500" if v > 20 else NEUTRAL) for v in miss]
ax.barh(range(len(miss)), miss.values, color=colors, height=0.8)
ax.set_yticks(range(len(miss)))
ax.set_yticklabels(miss.index, fontsize=6)
ax.axvline(50, color=ACCENT, ls="--", lw=1, label="50% threshold")
ax.axvline(20, color="#FFA500", ls=":", lw=1, label="20% threshold")
ax.set_xlabel("% missing")
ax.set_title("Missing-value rate – all 108 columns (sorted worst first)")
ax.legend()
plt.savefig(FIG_DIR / "notebook_02_plot_01.png", bbox_inches="tight")
plt.show()

print("\n=== Columns with >50% missing (candidate to drop) ===")
print(miss[miss > 50].to_string())
print(f"\nCount: {(miss > 50).sum()} columns > 50% missing")

In [ ]:
# ── missing-rate grouped by section ───────────────────────────────────────
section_order = ["group_01", "group_02", "group_03", "group_04"]

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=False)
for ax, sec in zip(axes, section_order):
    cols = channel.loc[channel["section"] == sec, "name"].tolist()
    cols = [c for c in cols if c in df.columns]
    m = df[cols].isna().mean() * 100
    bar_colors = [ACCENT if v > 50 else ("#FFA500" if v > 20 else NEUTRAL) for v in m]
    ax.barh(range(len(m)), m.values, color=bar_colors)
    ax.set_yticks(range(len(m)))
    ax.set_yticklabels(m.index, fontsize=6)
    ax.set_xlim(0, 105)
    ax.axvline(50, color=ACCENT, ls="--", lw=1)
    ax.set_title(f"{sec}\n({len(cols)} cols)", fontsize=8)
    ax.set_xlabel("% missing")

fig.suptitle("Missing-value rate by sensor section", fontsize=11, y=1.02)
plt.savefig(FIG_DIR / "notebook_02_plot_02.png", bbox_inches="tight")
plt.show()

In [ ]:
keep_cols = list(df.columns)
fig, ax = plt.subplots(figsize=(16, 7))
miss_mat = df[keep_cols].isna().astype(int).T
sns.heatmap(miss_mat, ax=ax, cmap="Reds", cbar=False,
            xticklabels=False, yticklabels=True)
ax.set_title("Missingness by anonymous channel")
plt.savefig(FIG_DIR / "notebook_02_plot_03.png", bbox_inches="tight")
plt.show()

### Analysis step 5

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 6

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# ── signal-based shutdown flag ───────────────────────────────────────────
SIGNAL_THRESH = 5.0   # units (both signals below this → shutdown)

shutdown_mask = (
    (df["channel_103"].fillna(0) < SIGNAL_THRESH) &
    (df["channel_102"].fillna(0) < SIGNAL_THRESH)
)

print(f"Shutdown days (signal-based): {shutdown_mask.sum()} / {len(df)}")
print(f"Operating days                : {(~shutdown_mask).sum()}")

# label contiguous shutdown blocks
blocks = shutdown_mask.astype(int).diff().fillna(0).abs().cumsum()
shutdown_events = []
for blk_id, grp in df[shutdown_mask].groupby(blocks[shutdown_mask]):
    shutdown_events.append({"start": grp.index.min(), "end": grp.index.max(),
                             "days": len(grp)})
sd = pd.DataFrame(shutdown_events)
print(f"\nShutdown blocks: {len(sd)}")
print(sd.to_string(index=False))

In [ ]:
# ── plot: top & bottom signal with shutdown bands ────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, col, label in [
    (ax1, "channel_103",  "Top signal"),
    (ax2, "channel_102", "Bottom signal"),
]:
    ax.plot(df.index, df[col], color=NEUTRAL, lw=0.8, label=label)
    for _, ev in sd.iterrows():
        ax.axvspan(ev["start"], ev["end"], color=ACCENT, alpha=0.25, lw=0)
    ax.axhline(SIGNAL_THRESH, color=GREY, ls="--", lw=1, label=f"Threshold {SIGNAL_THRESH} units")
    ax.set_ylabel(f"{label} (units)")
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

fig.suptitle("asset Signals — red shading = shutdown periods", fontsize=11)
plt.savefig(FIG_DIR / "notebook_02_plot_04.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── calendar heatmap of shutdowns ─────────────────────────────────────────
# Simple month × day-of-month grid
df_cal = df[["channel_103"]].copy()
df_cal["shutdown"] = shutdown_mask.astype(int)
df_cal["month"] = df_cal.index.to_period("M")
df_cal["day"] = df_cal.index.day

pivot = df_cal.pivot_table(index="month", columns="day", values="shutdown", aggfunc="max")

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    pivot, ax=ax, cmap="Reds", linewidths=0.3, linecolor="#ddd",
    cbar_kws={"label": "Shutdown (1=yes)"},
    vmin=0, vmax=1
)
ax.set_title("Shutdown calendar (red = offline day)", fontsize=10)
ax.set_xlabel("Day of month")
ax.set_ylabel("Month")
plt.savefig(FIG_DIR / "notebook_02_plot_05.png", bbox_inches="tight")
plt.show()

### Analysis step 7

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 8

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# operating-days-only slice
op = df[~shutdown_mask].copy()
print(f"Operating days: {len(op)}")

core_channels = [
    "channel_098", "channel_103", "channel_102",
    "channel_105", "channel_106", "channel_107",
    "channel_101", "channel_108", "channel_100",
    "channel_104", "channel_095",
    "channel_094", "channel_093",
    "channel_099",
]
signal_labels = {
    "channel_098":               "asset signal (differential signal)",
    "channel_103":         "asset Top signal",
    "channel_102":        "asset Bottom signal",
    "channel_105":          "asset Top signal",
    "channel_106":        "asset Upper signal",
    "channel_107":          "asset Mid signal",
    "channel_101":    "asset Last segment signal",
    "channel_108":   "asset Above-bottom signal",
    "channel_100":         "asset Bottom signal",
    "channel_104":      "asset signal level",
    "channel_095":              "asset Signal (condenser / signal side)",
    "channel_094":              "asset signal pump flow",
    "channel_093": "asset → GD-401 mixer flow",
    "channel_099":                "signal  in output",
}

### Analysis step 9

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(op.index, op["channel_098"], color=NEUTRAL, lw=0.9, label="asset signal (raw)")

# 14-day trailing mean
rolling_signal = op["channel_098"].rolling(14, min_periods=5).mean()
ax.plot(op.index, rolling_signal, color=ACCENT, lw=1.5, label="14-day rolling mean")

# highlight shutdown bands
for _, ev in sd.iterrows():
    ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.35, lw=0, label="_")

ax.axhline(0, color="black", ls="--", lw=0.7)
ax.set_ylabel("signal")
ax.set_title("asset Differential Signal (signal) — operating days only, with 14-day rolling mean\n"
             "Grey = shutdown periods")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.savefig(FIG_DIR / "notebook_02_plot_06.png", bbox_inches="tight")
plt.show()

# summary stats by half-year
op_signal = op["channel_098"].dropna()
print("channel stats by 6-month period (operating days only):")
print(
    op_signal.groupby(op_signal.index.to_period("Q"))
          .agg(["mean", "std", "min", "max"])
          .round(4)
          .to_string()
)

### Analysis step 10

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
n_cols = len(core_channels)
ncols_grid = 2
nrows_grid = (n_cols + ncols_grid - 1) // ncols_grid

fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(16, nrows_grid * 3), sharex=True)
axes = axes.flatten()

for ax, col in zip(axes, core_channels):
    ax.plot(op.index, op[col], color=NEUTRAL, lw=0.7, alpha=0.85)
    # 14-day rolling mean
    ax.plot(op.index, op[col].rolling(14, min_periods=5).mean(),
            color=ACCENT, lw=1.3)
    for _, ev in sd.iterrows():
        ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.3, lw=0)
    ax.set_title(signal_labels.get(col, col), fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)

for ax in axes[n_cols:]:
    ax.set_visible(False)

fig.suptitle("asset Asset Sensors (operating days) — blue=raw, red=14-day mean, grey=shutdown",
             fontsize=10, y=1.01)
plt.savefig(FIG_DIR / "notebook_02_plot_07.png", bbox_inches="tight")
plt.show()

### Analysis step 11

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# raw + rolling
ax = axes[0]
ax.plot(op.index, op["channel_099"], color=NEUTRAL, lw=0.7, label="signal  (raw)")
ax.plot(op.index, op["channel_099"].rolling(14, min_periods=5).mean(),
        color=ACCENT, lw=1.4, label="14-day rolling mean")
for _, ev in sd.iterrows():
    ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.3, lw=0)
ax.set_ylabel("signal ()")
ax.set_title("signal  in output — raw and 14-day mean")
ax.legend()

# log scale to reveal earlier variation
ax2 = axes[1]
signal_pos = op["channel_099"].clip(lower=0.01)   # avoid log(0)
ax2.semilogy(op.index, signal_pos, color=NEUTRAL, lw=0.7)
ax2.semilogy(op.index, signal_pos.rolling(14, min_periods=5).mean(),
             color=ACCENT, lw=1.4)
for _, ev in sd.iterrows():
    ax2.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.3, lw=0)
ax2.set_ylabel("signal (, log scale)")
ax2.set_title("Same — log scale to see early variation")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

plt.savefig(FIG_DIR / "notebook_02_plot_08.png", bbox_inches="tight")
plt.show()

print("signal  — quarterly stats (operating days):")
c3 = op["channel_099"].dropna()
print(
    c3.groupby(c3.index.to_period("Q"))
      .agg(["mean", "median", "std", "max"])
      .round(1)
      .to_string()
)

### Analysis step 12

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
n = len(core_channels)
ncols_g = 4
nrows_g = (n + ncols_g - 1) // ncols_g

fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(16, nrows_g * 3))
axes = axes.flatten()

sd_mask = shutdown_mask
for ax, col in zip(axes, core_channels):
    data_op  = df.loc[~sd_mask, col].dropna()
    data_sd  = df.loc[sd_mask, col].dropna()
    ax.hist(data_op,  bins=40, color=NEUTRAL, alpha=0.7, label="Operating", density=True)
    ax.hist(data_sd,  bins=20, color=ACCENT,  alpha=0.6, label="Shutdown",  density=True)
    ax.set_title(col.replace("channel_", "").replace("_", " "), fontsize=7)
    ax.legend(fontsize=5)

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle("Distribution of asset sensors: operating (blue) vs shutdown (red)", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_09.png", bbox_inches="tight")
plt.show()

### Analysis step 13

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
key_sensors = ["channel_098", "channel_103", "channel_102",
               "channel_107", "channel_104", "channel_099"]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=False)
axes = axes.flatten()

for ax, col in zip(axes, key_sensors):
    for yr, color in [(2032, NEUTRAL), (2033, ACCENT)]:
        sub = op[op.index.year == yr][col].dropna()
        # convert to day-of-year for overlay
        doy = sub.index.dayofyear
        ax.plot(doy, sub.values, color=color, alpha=0.6, lw=0.8, label=str(yr))
    ax.set_title(signal_labels.get(col, col), fontsize=8)
    ax.set_xlabel("Day of year")
    ax.legend(fontsize=7)

fig.suptitle("Year-over-year overlay: 2032 (blue) vs 2033 (red)", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_10.png", bbox_inches="tight")
plt.show()

## Analysis step 14

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
context_cols = ["channel_001", "channel_004", "channel_002",
             "channel_003", "channel_070", "channel_096", "channel_097"]

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for ax, col in zip(axes, context_cols):
    ax.plot(op.index, op[col], color="#2ca02c", lw=0.7, alpha=0.8)
    ax.plot(op.index, op[col].rolling(14, min_periods=5).mean(),
            color="#1a7a1a", lw=1.3)
    for _, ev in sd.iterrows():
        ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.3, lw=0)
    ax.set_title(col.replace("_", " "), fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)

axes[-1].set_visible(False)
fig.suptitle("Context Composition Sensors (operating days, green=raw, dark=14-day mean)",
             fontsize=10, y=1.01)
plt.savefig(FIG_DIR / "notebook_02_plot_11.png", bbox_inches="tight")
plt.show()

In [ ]:
# stacked area chart – context composition proportions over time (daily)
comp_cols = ["channel_001", "channel_004", "channel_002", "channel_003"]
comp = op[comp_cols].copy().rolling(7, min_periods=3).mean().dropna()
total = comp.sum(axis=1)
comp_norm = comp.divide(total, axis=0) * 100   # normalise to 100% for stacked area

fig, ax = plt.subplots(figsize=(14, 4))
comp_norm.plot.area(ax=ax, alpha=0.75,
                    color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])
ax.set_ylabel("Normalized value")
ax.set_title("group_01 (7-day mean, normalised) — do proportions shift before failure?")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.savefig(FIG_DIR / "notebook_02_plot_12.png", bbox_inches="tight")
plt.show()

## Analysis step 15

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
group_signal_channels = ['channel_063', 'channel_064', 'channel_065', 'channel_066', 'channel_067', 'channel_068', 'channel_069']
fig, ax = plt.subplots(figsize=(14, 4))
colors_7 = plt.cm.tab10.colors[:7]
for i, col in enumerate(group_signal_channels):
    ax.plot(op.index, op[col].rolling(7, min_periods=3).mean(), color=colors_7[i], lw=1.0, label=f"Group {i + 1}")
ax.set_title("Group signal traces (7-day rolling mean)")
ax.set_ylabel("Scaled signal")
ax.legend(ncol=4, fontsize=7)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.savefig(FIG_DIR / "notebook_02_plot_13.png", bbox_inches="tight")
plt.show()

In [ ]:
flow_channels = ['channel_005', 'channel_006', 'channel_007', 'channel_008', 'channel_009', 'channel_010', 'channel_011']
fig, ax = plt.subplots(figsize=(14, 4))
for i, col in enumerate(flow_channels):
    ax.plot(op.index, op[col].rolling(7, min_periods=3).mean(), color=colors_7[i], lw=0.9, label=f"Group {i + 1}")
ax.set_title("Group signal trends (7-day rolling mean)")
ax.set_ylabel("Scaled signal")
ax.legend(ncol=4, fontsize=7)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.savefig(FIG_DIR / "notebook_02_plot_14.png", bbox_inches="tight")
plt.show()

In [ ]:
controller_channels = ['channel_019', 'channel_020', 'channel_021', 'channel_022', 'channel_023', 'channel_024', 'channel_025']
fig, axes = plt.subplots(1, len(controller_channels), figsize=(16, 3))
for i, (ax, col) in enumerate(zip(axes, controller_channels)):
    ax.hist(op[col].dropna(), bins=30, color=NEUTRAL)
    ax.set_title(f"Group {i + 1}", fontsize=8)
    ax.set_xlabel("Signal", fontsize=7)
fig.suptitle("Generic group signal distributions", fontsize=9)
plt.savefig(FIG_DIR / "notebook_02_plot_15.png", bbox_inches="tight")
plt.show()

### Analysis step 16

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 17

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
group_channels = [f"channel_{i:03d}" for i in range(72, 79)]
n = len(group_channels)
ncols_g = 3
nrows_g = (n + ncols_g - 1) // ncols_g
fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(15, nrows_g * 3), sharex=True, squeeze=False)
axes = axes.flatten()
for ax, col in zip(axes, group_channels):
    ax.plot(op.index, op[col], color=NEUTRAL, lw=0.7, alpha=0.8)
    ax.plot(op.index, op[col].rolling(14, min_periods=5).mean(), color=ACCENT, lw=1.3)
    ax.set_title(col.replace("_", " "), fontsize=7)
for ax in axes[n:]:
    ax.set_visible(False)
fig.suptitle("Anonymous channel group trends", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_16.png", bbox_inches="tight")
plt.show()

In [ ]:
subset_a = [f"channel_{i:03d}" for i in range(72, 75)]
subset_b = [f"channel_{i:03d}" for i in range(75, 79)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, channels, label in [
    (axes[0], subset_a, "Channel subset A"),
    (axes[1], subset_b, "Channel subset B"),
]:
    for col in channels:
        ax.plot(op.index, op[col], lw=0.8, alpha=0.7, label=col)
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=7, ncol=2)
fig.suptitle("Anonymous channel subset comparison", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_17.png", bbox_inches="tight")
plt.show()

## Analysis step 18

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# IQR-based outlier count for the usable columns
usable_cols = core_channels + context_cols + group_channels
usable_cols = [c for c in usable_cols if c in op.columns]

outlier_stats = []
for col in usable_cols:
    s = op[col].dropna()
    if len(s) < 10:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    n_out = int(((s < lo) | (s > hi)).sum())
    outlier_stats.append({
        "column": col,
        "n_values": len(s),
        "n_outliers_3IQR": n_out,
        "pct_outliers": round(100 * n_out / len(s), 2),
        "min": round(s.min(), 3),
        "max": round(s.max(), 3),
    })

out_df = pd.DataFrame(outlier_stats).sort_values("pct_outliers", ascending=False)
print("=== Columns with >1% IQR-3 outliers ===")
print(out_df[out_df["pct_outliers"] > 1].to_string(index=False))

In [ ]:
# Box plots – asset asset sensors (operating days)
fig, ax = plt.subplots(figsize=(14, 5))
box_data = [op[col].dropna().values for col in core_channels]
bp = ax.boxplot(box_data, patch_artist=True, medianprops=dict(color=ACCENT, lw=1.5),
                flierprops=dict(marker=".", markersize=2, color=ACCENT, alpha=0.5))
for patch in bp["boxes"]:
    patch.set_facecolor(NEUTRAL)
    patch.set_alpha(0.4)
ax.set_xticks(range(1, len(core_channels) + 1))
ax.set_xticklabels([c.replace("channel_", "").replace("channel_", "").replace("_", "\n")
                    for c in core_channels], fontsize=7)
ax.set_title("asset asset sensor box plots (operating days) — outliers shown as dots")
plt.savefig(FIG_DIR / "notebook_02_plot_18.png", bbox_inches="tight")
plt.show()

In [ ]:
# Negative-value check for physically-impossible negatives
neg_check = ["channel_098", "channel_103", "channel_102",
             "channel_104", "channel_094", "channel_093"]

print("Negative-value counts on operating days:")
for col in neg_check:
    s = op[col].dropna()
    n_neg = (s < 0).sum()
    print(f"  {col:35s}  n_neg={n_neg:4d}  min={s.min():.4f}")

## Analysis step 19

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Correlation heatmap — asset asset sensors
corr = op[core_channels].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    linewidths=0.4, square=True
)
ax.set_title("Pearson correlation — asset asset sensors (operating days)", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_19.png", bbox_inches="tight")
plt.show()

# print highly-correlated pairs
corr_pairs = (
    corr.where(mask == False)
        .stack()
        .reset_index()
        .rename(columns={"level_0": "col_A", "level_1": "col_B", 0: "r"})
)
print("\nPairs with |r| > 0.85 (potential duplicates):")
print(corr_pairs[corr_pairs["r"].abs() > 0.85].sort_values("r", ascending=False).to_string(index=False))

In [ ]:
# Broader correlation: all usable columns (context + asset + group_03)
broad_cols = [c for c in (core_channels + context_cols) if c in op.columns]
corr_broad = op[broad_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask_b = np.triu(np.ones_like(corr_broad, dtype=bool))
sns.heatmap(
    corr_broad, mask=mask_b, ax=ax,
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True, fmt=".1f", annot_kws={"size": 6},
    linewidths=0.3
)
ax.set_title("Correlation — asset asset + context composition (operating days)", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_20.png", bbox_inches="tight")
plt.show()

In [ ]:
# signal group signal signal correlation with each other (check if they move together)
group_signal_cols = ['channel_063', 'channel_064', 'channel_065', 'channel_066', 'channel_067', 'channel_068', 'channel_069']
corr_group_signal = op[group_signal_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr_group_signal, ax=ax, cmap="Greens", annot=True, fmt=".2f",
            annot_kws={"size": 9}, square=True, vmin=0, vmax=1)
ax.set_title("signal group signal signal correlation matrix")
plt.savefig(FIG_DIR / "notebook_02_plot_21.png", bbox_inches="tight")
plt.show()

## Analysis step 20

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Zoom into 2033 for the key asset sensors
op_2033 = op.loc["2033":]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

zoom_sensors = [
    ("channel_098",            "signal — rising signal = gradual drift / gradual drift risk"),
    ("channel_099",             "signal  — rising = relationship degradation"),
    ("channel_107",       "Mid-column signal"),
    ("channel_104",   "signal level"),
    ("channel_103",      "Top signal"),
    ("channel_094",           "asset signal pump flow"),
]

for ax, (col, title) in zip(axes, zoom_sensors):
    ax.plot(op_2033.index, op_2033[col], color=NEUTRAL, lw=0.8, alpha=0.8)
    ax.plot(op_2033.index, op_2033[col].rolling(14, min_periods=5).mean(),
            color=ACCENT, lw=1.5)
    for _, ev in sd[sd["start"] >= pd.Timestamp("2033-01-01")].iterrows():
        ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.35, lw=0)
    ax.set_title(title, fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha="right", fontsize=7)

fig.suptitle("2033 sensor behaviour — looking for failure drift (red = 14-day mean)",
             fontsize=10, y=1.01)
plt.savefig(FIG_DIR / "notebook_02_plot_22.png", bbox_inches="tight")
plt.show()

In [ ]:
# signal distribution: 2032 healthy vs 2033 (split by quarter)
fig, ax = plt.subplots(figsize=(10, 5))

signal_2032 = op.loc["2032", "channel_098"].dropna()
quarters_2033 = {
    "2033-Q1": ("2033-01-01", "2033-03-31", "#AED6F1"),
    "2033-Q2": ("2033-04-01", "2033-06-30", "#85C1E9"),
    "2033-Q3": ("2033-07-01", "2033-09-30", "#5DADE2"),
    "2033-Q4": ("2033-10-01", "2033-12-31", ACCENT),
}

ax.hist(signal_2032, bins=40, alpha=0.55, color=NEUTRAL, density=True, label="2032 (healthy ref)")
for label, (start, end, col) in quarters_2033.items():
    signal_q = op.loc[start:end, "channel_098"].dropna()
    if len(signal_q) > 5:
        ax.hist(signal_q, bins=25, alpha=0.6, color=col, density=True, label=label)

ax.set_xlabel("asset signal")
ax.set_ylabel("Density")
ax.set_title("signal distribution: 2032 baseline vs 2033 quarters — shift = degradation")
ax.legend()
plt.savefig(FIG_DIR / "notebook_02_plot_23.png", bbox_inches="tight")
plt.show()

In [ ]:
# Monthly signal box plots to see drift month by month
op_copy = op[["channel_098"]].copy()
op_copy["month"] = op_copy.index.to_period("M").astype(str)
op_copy = op_copy.dropna()

months_order = sorted(op_copy["month"].unique())
box_data_monthly = [op_copy.loc[op_copy["month"] == m, "channel_098"].values
                    for m in months_order]

fig, ax = plt.subplots(figsize=(16, 5))
bp = ax.boxplot(box_data_monthly, patch_artist=True,
                medianprops=dict(color=ACCENT, lw=1.5),
                flierprops=dict(marker=".", markersize=2, color=ACCENT))
for i, patch in enumerate(bp["boxes"]):
    yr = months_order[i][:4]
    patch.set_facecolor(NEUTRAL if yr == "2032" else "#FF9999")
    patch.set_alpha(0.5)
ax.set_xticks(range(1, len(months_order) + 1))
ax.set_xticklabels(months_order, rotation=45, ha="right", fontsize=7)
ax.axhline(0, color="black", ls="--", lw=0.7)
ax.set_ylabel("asset signal")
ax.set_title("Monthly signal box plots (blue=2032, red=2033) — track drift month by month")
plt.savefig(FIG_DIR / "notebook_02_plot_24.png", bbox_inches="tight")
plt.show()

In [ ]:
# signal  monthly box plot
op_c3 = op[["channel_099"]].copy().dropna()
op_c3["month"] = op_c3.index.to_period("M").astype(str)

months_order_c3 = sorted(op_c3["month"].unique())
box_data_c3 = [op_c3.loc[op_c3["month"] == m, "channel_099"].values
               for m in months_order_c3]

fig, ax = plt.subplots(figsize=(16, 5))
bp2 = ax.boxplot(box_data_c3, patch_artist=True,
                 medianprops=dict(color=ACCENT, lw=1.5),
                 flierprops=dict(marker=".", markersize=2, color=ACCENT))
for i, patch in enumerate(bp2["boxes"]):
    yr = months_order_c3[i][:4]
    patch.set_facecolor("#AED6F1" if yr == "2032" else "#FF9999")
    patch.set_alpha(0.5)
ax.set_xticks(range(1, len(months_order_c3) + 1))
ax.set_xticklabels(months_order_c3, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("signal ")
ax.set_title("Monthly signal  box plots (blue=2032, red=2033) — jump = relationship failure")
plt.savefig(FIG_DIR / "notebook_02_plot_25.png", bbox_inches="tight")
plt.show()

## Analysis step 21

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Scatter matrix of selected anonymous signals
scatter_cols = ["channel_098", "channel_105", "channel_107",
                "channel_108", "channel_100", "channel_099"]

op_scatter = op[scatter_cols].dropna().copy()
op_scatter["year"] = op_scatter.index.year

g = sns.pairplot(
    op_scatter, hue="year", diag_kind="kde",
    plot_kws={"alpha": 0.4, "s": 15},
    palette={2032: NEUTRAL, 2033: ACCENT},
    corner=True,
)
g.figure.suptitle("Scatter matrix of anonymous signals by synthetic year", y=1.02, fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_26.png", bbox_inches="tight")
plt.show()

In [ ]:
# signal vs signal scatter — the two key failure indicators
fig, ax = plt.subplots(figsize=(8, 6))
signal_c3 = op[["channel_098", "channel_099"]].dropna().copy()
signal_c3["year"] = signal_c3.index.year

for yr, color, label in [(2032, NEUTRAL, "2032"), (2033, ACCENT, "2033")]:
    sub = signal_c3[signal_c3["year"] == yr]
    ax.scatter(sub["channel_098"], sub["channel_099"],
               color=color, alpha=0.5, s=20, label=label)

ax.set_xlabel("asset signal")
ax.set_ylabel("signal ")
ax.set_title("signal vs signal  — rising together = gradual drift / gradual drift degradation")
ax.legend()
plt.savefig(FIG_DIR / "notebook_02_plot_26.png", bbox_inches="tight")
plt.show()

## Analysis step 22

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Gap length distribution for usable asset + context columns
priority_cols = core_channels + context_cols
priority_cols = [c for c in priority_cols if c in op.columns]

gap_stats = []
for col in priority_cols:
    s = op[col]
    is_missing = s.isna()
    # find gap lengths
    gap_lengths = []
    curr = 0
    for v in is_missing:
        if v:
            curr += 1
        else:
            if curr > 0:
                gap_lengths.append(curr)
            curr = 0
    if curr > 0:
        gap_lengths.append(curr)
    if gap_lengths:
        gap_stats.append({
            "column": col,
            "n_gaps": len(gap_lengths),
            "max_gap_days": max(gap_lengths),
            "mean_gap_days": round(np.mean(gap_lengths), 1),
        })

gap_df = pd.DataFrame(gap_stats).sort_values("max_gap_days", ascending=False)
print("=== Gap analysis for priority columns ===")
print(gap_df.to_string(index=False))

In [ ]:
# Plot: max gap per column
fig, ax = plt.subplots(figsize=(12, 5))
if len(gap_df) > 0:
    gap_colors = [ACCENT if v > 7 else ("#FFA500" if v > 3 else NEUTRAL) for v in gap_df["max_gap_days"]]
    ax.barh(range(len(gap_df)), gap_df["max_gap_days"], color=gap_colors)
    ax.set_yticks(range(len(gap_df)))
    ax.set_yticklabels(gap_df["column"], fontsize=8)
    ax.axvline(3,  color="#FFA500", ls=":",  lw=1.5, label="3-day fill limit")
    ax.axvline(7,  color=ACCENT,    ls="--", lw=1.5, label="7-day fill limit")
    ax.set_xlabel("Longest single gap (days)")
    ax.set_title("Maximum consecutive missing days per column (operating days only)")
    ax.legend()
plt.savefig(FIG_DIR / "notebook_02_plot_27.png", bbox_inches="tight")
plt.show()

## Analysis step 23

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# asset asset sensors: descriptive stats split by year
signal_stats_rows = []
for col in core_channels:
    for yr in [2032, 2033]:
        s = op.loc[op.index.year == yr, col].dropna()
        if len(s) == 0:
            continue
        signal_stats_rows.append({
            "column": col,
            "year": yr,
            "n": len(s),
            "mean": s.mean(),
            "std": s.std(),
            "p5": s.quantile(0.05),
            "median": s.median(),
            "p95": s.quantile(0.95),
        })

signal_stats = pd.DataFrame(signal_stats_rows).round(4)
print("=== asset asset sensor stats: 2032 vs 2033 ===")
print(signal_stats.to_string(index=False))

In [ ]:
# Visual: mean ± std per sensor per year (2032=blue, 2033=red)
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, core_channels):
    for yr, color in [(2032, NEUTRAL), (2033, ACCENT)]:
        s = op.loc[op.index.year == yr, col].dropna()
        if len(s) == 0:
            continue
        mu, sigma = s.mean(), s.std()
        ax.barh([str(yr)], [mu], xerr=[sigma], color=color, alpha=0.7,
                error_kw={"ecolor": "black", "lw": 1.5})
    ax.set_title(col.replace("channel_", "").replace("_", " "), fontsize=7)

for ax in axes[len(core_channels):]:
    ax.set_visible(False)

fig.suptitle("Mean ± std of asset sensors: 2032 (blue) vs 2033 (red)", fontsize=10)
plt.savefig(FIG_DIR / "notebook_02_plot_28.png", bbox_inches="tight")
plt.show()

## Analysis step 24

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Rolling standard deviation: does variability change before failure?
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

key_roll_cols = ["channel_098", "channel_099", "channel_107",
                 "channel_104", "channel_105", "channel_094"]

for ax, col in zip(axes, key_roll_cols):
    roll_std = op[col].rolling(30, min_periods=10).std()
    ax.plot(op.index, roll_std, color=NEUTRAL, lw=0.9)
    for _, ev in sd.iterrows():
        ax.axvspan(ev["start"], ev["end"], color=GREY, alpha=0.3, lw=0)
    ax.set_title(f"30-day rolling σ: {col.replace('channel_','').replace('_',' ')}",
                 fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)

fig.suptitle("Rolling standard deviation (30-day window) — does volatility increase before failure?",
             fontsize=10, y=1.01)
plt.savefig(FIG_DIR / "notebook_02_plot_29.png", bbox_inches="tight")
plt.show()

## Analysis step 25

This section uses synthetic values and anonymous channel identifiers.

### Analysis step 26

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
keep_drop = {
    # asset asset — all keep (core analysis)
    "channel_098":               "KEEP — primary failure indicator (differential signal)",
    "channel_103":         "KEEP — operating signal confirmation",
    "channel_102":        "KEEP — operating signal confirmation",
    "channel_105":          "KEEP — column top signal",
    "channel_106":        "KEEP — column profile",
    "channel_107":          "KEEP — column mid signal",
    "channel_101":    "KEEP — column stripping section",
    "channel_108":   "KEEP — column bottom section",
    "channel_100":         "KEEP — reboiler signal",
    "channel_104":      "KEEP — signal level (gradual drift early sign)",
    "channel_095":              "KEEP — condenser performance",
    "channel_094":              "KEEP — signal flow",
    "channel_093": "KEEP — context to asset",
    "channel_099":                "KEEP — output quality / relationship failure indicator",
    # group_01 — keep
    "channel_001":        "KEEP — context context",
    "channel_004":         "KEEP — context context (primary component)",
    "channel_002":        "KEEP — context context",
    "channel_003":            "KEEP — context context",
    "channel_070":              "REVIEW — unit unknown, ask system",
    "channel_096":              "KEEP — context context",
    "channel_097":                "KEEP — context context",
    # signal — selective keep
    "channel_005":         "KEEP (context) — total context to system",
    "channel_063":          "KEEP (context) — cracking severity",
    "channel_012":     "KEEP (context) — check negatives",
    "channel_019": "DROP — binary flag (ON/OFF), not a sensor",
    "channel_026":        "DROP — binary flag (0/0.5)",
    "channel_033":           "DROP — >59% missing, IQ calculation noise",
    "channel_040":             "DROP — >59% missing, IQ calculation noise",
    "channel_047": "DROP — >70% missing, boolean masking",
    "channel_055": "DROP — >70% missing, boolean masking",
    "channel_054": "DROP — >63% missing",
    "channel_062": "DROP — >63% missing",
    # asset — keep with note
    "channel_071":           "KEEP — signal inlet signal",
    "channel_072":                "KEEP — signal/signal ratio in signal",
    "channel_073":      "KEEP — signal  (outlet check)",
    "channel_075":               "KEEP — CO flow (catalyst indicator)",
    "channel_079":          "KEEP — bed B signal profile",
    "channel_086":        "FLAG — very wide range (7-396); check for hotspot events",
}

print(f"{'COLUMN':<40} {'DECISION'}")
print("-" * 90)
for col, decision in keep_drop.items():
    print(f"{col:<40} {decision}")

### Analysis step 27

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
# Save a summary of keep/drop decisions as CSV for notebook 03
kd_rows = [{"name": col, "eda_decision": dec} for col, dec in keep_drop.items()]
kd_df = pd.DataFrame(kd_rows)
out_path = ROOT / "data" / "interim" / "channel_selection.csv"
kd_df.to_csv(out_path, index=False)
print(f"Saved EDA keep/drop decisions → {out_path}")
print(f"\nFigures saved in: {FIG_DIR}")
print("\n=== Done — proceed to notebook 03_cleaning_and_shutdowns ===")